# Writing indexed CNDB files

OpenMiChroM can write CNDB files with a metadata header and embedded HDF5 object index. These files keep the familiar local CNDB layout (`/types`, `/0`, `/1`, ...) and can also be streamed remotely with `CNDBTools.from_remote` when hosted by an HTTP server that supports Range requests.

This tutorial writes a tiny synthetic trajectory, inspects the new `/Header` and `/_index`, serves the file from a local HTTP Range server, and streams a small bead subset back without downloading the full file.

## Imports

The synthetic example uses `SaveStructure` directly with a small fake OpenMM-like state object. In a normal simulation, `sim.createReporters(traj=True, trajFormat="cndb")` creates the reporter for you.

In [ ]:
import functools
import os
import re
import tempfile
import threading
from pathlib import Path
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer

import h5py
import numpy as np

from OpenMiChroM.CustomReporter import SaveStructure
from OpenMiChroM.CndbTools import CndbTools

## Write a tiny CNDB v2 file

The important options are `indexed=True` and `metadata=True`. They are the defaults for new CNDB files. Calling `close()` finalizes `n_frames`, writes `/_index`, and records `_index_offset`.

In [ ]:
class FakePositions:
    def __init__(self, data):
        self.data = np.asarray(data)

    def value_in_unit(self, unit):
        return self.data


class FakeState:
    def __init__(self, data):
        self.data = np.asarray(data)

    def getPositions(self, asNumpy=True):
        return FakePositions(self.data)


workdir = Path(tempfile.mkdtemp(prefix="openmichrom-cndb-v2-"))
frames = [
    np.arange(30, dtype=np.float32).reshape(10, 3),
    100 + np.arange(30, dtype=np.float32).reshape(10, 3),
]

reporter = SaveStructure(
    filePrefix="toy",
    reportInterval=1,
    mode="cndb",
    folder=str(workdir),
    chains=[(0, 9, False)],
    typeListLetter=np.array([b"A1", b"B1"] * 5),
    indexed=True,
    metadata=True,
    coordinate_dtype=np.float32,
)

for frame in frames:
    reporter.report(None, FakeState(frame))
reporter.close()

cndb_path = workdir / "toy_0.cndb"
print(cndb_path)

## Inspect the header and embedded index

The coordinate frames are still root-level numeric datasets. The new streamability metadata lives in `/Header`, `/_index`, and the root `_index_offset` attribute.

In [ ]:
with h5py.File(cndb_path, "r") as h5:
    print("Root keys:", list(h5.keys()))
    print("_index_offset:", int(h5.attrs["_index_offset"]))
    for key in ["format_name", "format_version", "n_beads", "n_frames", "indexed", "index_format"]:
        print(f"{key}:", h5["Header"].attrs[key])

## Read locally with CNDBTools

Local CNDBTools behavior is unchanged. The extra metadata and index objects are ignored by local coordinate reads.

In [ ]:
local_tools = CndbTools.open(str(cndb_path))
local_xyz = local_tools.xyz(frames=[0], beadSelection=range(0, 5))
print(local_xyz.shape)
np.testing.assert_allclose(local_xyz[0], frames[0][:5])

## Serve the file through a local HTTP Range server

This server is only for the tutorial. It returns `206 Partial Content` for valid Range requests, which is required for safe remote CNDB streaming.

In [ ]:
class RangeRequestHandler(SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        return

    def send_head(self):
        path = self.translate_path(self.path)
        if not os.path.isfile(path):
            self.send_error(404, "File not found")
            return None
        file_size = os.path.getsize(path)
        range_header = self.headers.get("Range")
        handle = open(path, "rb")
        if range_header:
            match = re.match(r"bytes=(\d+)-(\d+)$", range_header)
            if match is None:
                handle.close()
                self.send_error(416, "Invalid Range")
                return None
            start = int(match.group(1))
            stop = min(int(match.group(2)), file_size - 1)
            length = stop - start + 1
            self.send_response(206)
            self.send_header("Content-Type", "application/octet-stream")
            self.send_header("Accept-Ranges", "bytes")
            self.send_header("Content-Range", f"bytes {start}-{stop}/{file_size}")
            self.send_header("Content-Length", str(length))
            self.end_headers()
            handle.seek(start)
            self.range_length = length
            return handle
        self.send_response(200)
        self.send_header("Content-Type", "application/octet-stream")
        self.send_header("Accept-Ranges", "bytes")
        self.send_header("Content-Length", str(file_size))
        self.end_headers()
        self.range_length = None
        return handle

    def copyfile(self, source, outputfile):
        if self.range_length is None:
            return super().copyfile(source, outputfile)
        remaining = self.range_length
        while remaining:
            chunk = source.read(min(65536, remaining))
            if not chunk:
                break
            outputfile.write(chunk)
            remaining -= len(chunk)


handler = functools.partial(RangeRequestHandler, directory=str(workdir))
server = ThreadingHTTPServer(("127.0.0.1", 0), handler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
base_url = f"http://{server.server_address[0]}:{server.server_address[1]}"
print(base_url)

## Stream a small bead range remotely

The file is opened through HTTP, but the coordinate read fetches only the requested bead rows. For 5 beads and float32 coordinates, the coordinate payload is `5 × 3 × 4 = 60` bytes.

In [ ]:
remote_tools = CndbTools.from_remote(h5_url=f"{base_url}/{cndb_path.name}")
remote_xyz = remote_tools.xyz(frames=[0], beadSelection=range(0, 5))
print(remote_xyz.shape)
print(remote_tools.stream_stats())
np.testing.assert_allclose(remote_xyz[0], frames[0][:5])
assert remote_tools.stream_stats()["data_bytes_read"] == 5 * 3 * np.dtype("float32").itemsize

## Clean up the local server

In [ ]:
server.shutdown()
server.server_close()
thread.join(timeout=5)